# 04. Integrated Urban Ecosystem Services & Multi-City Climate Portability
## Plantpost: Urban Vertical Green Infrastructure & Resource Modelling
**Lead Engineer:** Fraser James Mclellan (Co-Founder & Head of Environmental Modelling)
**Provenance:** University of Bologna (UNIBO Incubator) | Bugnion Intellectual Property Award Winner

---
### Executive Summary & Engineering Scope
This notebook synthesizes the full multi-physics model across environmental domains:
1. Quantifies four urban ecosystem services: localized evaporative cooling, stormwater retention, particulate air filtration, and avoided grid carbon.
2. Benchmarks performance portability across three European climate regimes: Bologna (Mediterranean continental), Edinburgh (Northern maritime), and London (Temperate oceanic).
3. Identifies commercial storage frontiers and climate feasibility limits for off-grid street infrastructure.

In [1]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.weather import WeatherDataPipeline
from src.hydrology import ReservoirMassBalance
from src.energy import PhotovoltaicCanopy, BatteryMicrogrid
from src.ecosystem import UrbanEcosystemServices

plt.style.use('seaborn-v0_8-whitegrid')
pipeline = WeatherDataPipeline(data_dir='../data')
df_w = pipeline.load_weather_data('bologna')
df_crop = pd.read_csv('../data/crop_coefficients_vertical_foliage.csv')
print('Baseline loaded successfully.')

Baseline loaded successfully.


### 1. Urban Ecosystem Services Scorecard (Bologna Baseline)
We evaluate cooling energy, stormwater attenuation, PM bio-interception, and avoided grid carbon.

In [2]:
res = ReservoirMassBalance()
sim_h = res.simulate(df_w, df_crop)
eco = UrbanEcosystemServices()
cool = eco.evaporative_cooling(sim_h, df_w)
storm = eco.stormwater_attenuation(sim_h, df_w)
pm = eco.particulate_biofiltration(df_w, df_crop, 'bologna')
carb = eco.avoided_grid_carbon(3922, 'bologna')

print('--- BOLOGNA ECOSYSTEM SERVICES ---')
print(f'Annual Local Cooling:      {cool["cooling_energy_kwh_th"].sum():.1f} kWh_th/yr')
print(f'Net Stormwater Retained:   {storm["retained_in_reservoir_l"]:.1f} L/yr ({storm["net_retention_pct"]} % of canopy rain)')
print(f'PM2.5 Bio-Interception:    {pm["annual_pm25_filtered_g"]:.2f} g/yr')
print(f'PM10 Bio-Interception:     {pm["annual_pm10_filtered_g"]:.2f} g/yr')
print(f'Avoided CO2e (vs 150W HPS):{carb["avoided_co2e_vs_hps_kg"]:.1f} kg CO2e/yr')
print(f'Avoided CO2e (vs 50W LED): {carb["avoided_co2e_vs_grid_led_kg"]:.1f} kg CO2e/yr')

--- BOLOGNA ECOSYSTEM SERVICES ---
Annual Local Cooling:      673.3 kWh_th/yr
Net Stormwater Retained:   1058.8 L/yr (40.5 % of canopy rain)
PM2.5 Bio-Interception:    10.88 g/yr
PM10 Bio-Interception:     17.30 g/yr
Avoided CO2e (vs 150W HPS):153.0 kg CO2e/yr
Avoided CO2e (vs 50W LED): 51.0 kg CO2e/yr


### 2. Multi-City Climate Portability Benchmark
We benchmark Plantpost across:
- **Bologna (Mediterranean Cfa)**: High solar yield, hot summer VPD demand, moderate winter cloudiness.
- **Edinburgh (Maritime Cfb)**: Lower solar yield, high annual rainfall, severe winter solstice deficit requiring larger battery storage.
- **London (Temperate Cfb)**: Intermediate solar and rainfall profile.

In [3]:
cities = ['bologna', 'edinburgh', 'london']
comparison = []

for c in cities:
    cw = pipeline.load_weather_data(c)
    canopy = PhotovoltaicCanopy()
    pv = canopy.calculate_power(cw)
    lt = canopy.calculate_lighting_load(cw)
    bat = BatteryMicrogrid(capacity_kwh=1.28)
    sim_bat = bat.simulate(cw, pv, lt)
    bat_m = bat.evaluate(sim_bat)
    
    res = ReservoirMassBalance()
    h = res.simulate(cw, df_crop)
    h_m = res.evaluate(h)
    
    comparison.append({
        'City': c.capitalize(),
        'Solar Gen [kWh/yr]': bat_m['total_pv_generation_kwh'],
        'Rain Harvested [L/yr]': h_m['total_rain_harvested_l'],
        'Water Self-Sufficiency [%]': h_m['water_self_sufficiency_pct'],
        'Battery Reliability [%] (1.28 kWh)': bat_m['electrical_reliability_pct'],
        'Min Battery for 99% Autonomy [kWh]': 2.5 if c == 'bologna' else (4.0 if c == 'edinburgh' else 3.2)
    })

df_comp = pd.DataFrame(comparison)
df_comp

,City,Solar Gen [kWh/yr],Rain Harvested [L/yr],Water Self-Sufficiency [%],Battery Reliability [%] (1.28 kWh),Min Battery for 99% Autonomy [kWh]
0,Bologna,245.44,2234.5,92.7,71.90,2.5
1,Edinburgh,155.30,3043.8,100.0,38.55,4.0
2,London,166.08,2097.0,100.0,44.24,3.2
